In [1]:
# Cell 0: Setup
!pip install yfinance shap captum torch scikit-learn pandas numpy matplotlib --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
import time
import copy

print("Setup complete")

Setup complete


In [2]:
# Cell 1 (retry-safe version): Fetch with retry logic for failed tickers
import time

def fetch_with_retry(ticker, max_retries=3, delay=3):
    for attempt in range(max_retries):
        try:
            raw = yf.download(tickers=ticker, period="730d", interval="1h", progress=False)
            if raw is not None and len(raw) > 0:
                return raw
        except Exception as e:
            print(f"  Attempt {attempt+1} failed for {ticker}: {e}")
        time.sleep(delay)  # wait before retrying
    return None

tickers = ["AAPL", "MSFT", "AMZN", "GOOG", "META", "NVDA", "TSLA", "NFLX", "AMD", "INTC"]
all_dfs = []

for t in tickers:
    raw = fetch_with_retry(t)
    if raw is None or len(raw) == 0:
        print(f"{t}: FAILED after retries, skipping")
        continue

    d = raw.reset_index()
    d.columns = [c[0] if isinstance(c, tuple) else c for c in d.columns]
    first_col = d.columns[0]
    d = d.rename(columns={
        first_col: "open_time",
        "Open": "open", "High": "high", "Low": "low",
        "Close": "close", "Volume": "volume"
    })
    d = d[["open_time", "open", "high", "low", "close", "volume"]]
    d["ticker"] = t
    all_dfs.append(d)
    print(f"{t}: {d.shape[0]} rows")

    time.sleep(1.5)  # small pause between tickers to avoid rate-limiting

df_pooled = pd.concat(all_dfs, ignore_index=True).sort_values(["ticker", "open_time"]).reset_index(drop=True)
print("\nTotal pooled shape:", df_pooled.shape)

AAPL: 5082 rows
MSFT: 5082 rows
AMZN: 5082 rows
GOOG: 5082 rows
META: 5082 rows
NVDA: 5082 rows
TSLA: 5082 rows
NFLX: 5082 rows
AMD: 5082 rows
INTC: 5082 rows

Total pooled shape: (50820, 7)


In [4]:
# Cell 2: Confirm which tickers actually made it into the pooled data
successful_tickers = df_pooled["ticker"].unique().tolist()
print("Successfully fetched tickers:", successful_tickers)
tickers = successful_tickers  # update tickers list to match what we actually have

Successfully fetched tickers: ['AAPL', 'AMD', 'AMZN', 'GOOG', 'INTC', 'META', 'MSFT', 'NFLX', 'NVDA', 'TSLA']


In [6]:
# NEW CELL 3 — insert before Cell 4
def add_indicators_v2(d):
    d = d.copy()

    d["sma_12"] = d["close"].rolling(window=12).mean()
    d["sma_48"] = d["close"].rolling(window=48).mean()

    delta = d["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    d["rsi_14"] = 100 - (100 / (1 + rs))

    d["volatility_24"] = d["close"].pct_change().rolling(window=24).std()
    d["momentum_6"] = d["close"].pct_change(periods=6)

    ema_12 = d["close"].ewm(span=12, adjust=False).mean()
    ema_26 = d["close"].ewm(span=26, adjust=False).mean()
    d["macd"] = ema_12 - ema_26
    d["macd_signal"] = d["macd"].ewm(span=9, adjust=False).mean()
    d["macd_hist"] = d["macd"] - d["macd_signal"]

    sma_20 = d["close"].rolling(window=20).mean()
    std_20 = d["close"].rolling(window=20).std()
    d["bb_upper"] = sma_20 + (2 * std_20)
    d["bb_lower"] = sma_20 - (2 * std_20)
    d["bb_width"] = (d["bb_upper"] - d["bb_lower"]) / (sma_20 + 1e-9)
    d["bb_position"] = (d["close"] - d["bb_lower"]) / (d["bb_upper"] - d["bb_lower"] + 1e-9)

    high_low = d["high"] - d["low"]
    high_close = (d["high"] - d["close"].shift()).abs()
    low_close = (d["low"] - d["close"].shift()).abs()
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    d["atr_14"] = true_range.rolling(window=14).mean()

    low_14 = d["low"].rolling(window=14).min()
    high_14 = d["high"].rolling(window=14).max()
    d["stoch_k"] = 100 * (d["close"] - low_14) / (high_14 - low_14 + 1e-9)
    d["stoch_d"] = d["stoch_k"].rolling(window=3).mean()

    return d.dropna().reset_index(drop=True)


def add_lag_and_time_features(d):
    d = d.copy()

    returns = d["close"].pct_change()
    d["lag_return_3h"] = returns.shift(1).rolling(window=3).sum()
    d["lag_return_6h"] = returns.shift(1).rolling(window=6).sum()
    d["lag_return_12h"] = returns.shift(1).rolling(window=12).sum()
    d["lag_return_24h"] = returns.shift(1).rolling(window=24).sum()

    d["hour_of_day"] = pd.to_datetime(d["open_time"]).dt.hour
    d["day_of_week"] = pd.to_datetime(d["open_time"]).dt.dayofweek

    d["hour_sin"] = np.sin(2 * np.pi * d["hour_of_day"] / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d["hour_of_day"] / 24)
    d["dow_sin"] = np.sin(2 * np.pi * d["day_of_week"] / 7)
    d["dow_cos"] = np.cos(2 * np.pi * d["day_of_week"] / 7)

    return d.dropna().reset_index(drop=True)

print("Feature engineering functions ready")

Feature engineering functions ready


In [7]:
# Cell 4: Preprocessing per ticker — expanded indicators + lag/time features, then combine
WINDOW = 72
all_X, all_y = [], []
ticker_counts = {}

base_feature_cols = [
    "open", "high", "low", "close",
    "sma_12", "sma_48", "rsi_14", "volatility_24", "momentum_6",
    "macd", "macd_signal", "macd_hist",
    "bb_width", "bb_position",
    "atr_14", "stoch_k", "stoch_d",
    "lag_return_3h", "lag_return_6h", "lag_return_12h", "lag_return_24h",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos"
]

for t in tickers:
    d = df_pooled[df_pooled["ticker"] == t].reset_index(drop=True)
    d = add_indicators_v2(d)
    d = add_lag_and_time_features(d)

    price_cols = ["open", "high", "low", "close"]
    other_cols = [c for c in base_feature_cols if c not in price_cols]

    df_returns = d[price_cols].pct_change()
    df_other = d[other_cols]
    df_combined = pd.concat([df_returns, df_other], axis=1).dropna().reset_index(drop=True)

    close_returns = d["close"].pct_change().dropna().reset_index(drop=True)
    data = df_combined.values

    n_windows = len(data) - WINDOW
    for i in range(n_windows):
        window = data[i : i + WINDOW]
        label = 1 if close_returns.iloc[i + WINDOW] > 0 else 0
        all_X.append(window)
        all_y.append(label)

    ticker_counts[t] = n_windows
    print(f"{t}: {n_windows} windows built")

X = np.array(all_X)
y = np.array(all_y)
print("\nFinal X shape:", X.shape, "y shape:", y.shape, "Up ratio:", y.mean())

AAPL: 4937 windows built
AMD: 4937 windows built
AMZN: 4937 windows built
GOOG: 4937 windows built
INTC: 4937 windows built
META: 4937 windows built
MSFT: 4937 windows built
NFLX: 4937 windows built
NVDA: 4937 windows built
TSLA: 4937 windows built

Final X shape: (49370, 72, 25) y shape: (49370,) Up ratio: 0.5103909256633583


In [8]:
# Cell 5 (fixed): Standardize BEFORE applying PCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

n_samples, n_steps, n_feat = X.shape
X_flat = X.reshape(-1, n_feat)

# Standardize first (critical step that was missing)
pre_pca_scaler = StandardScaler()
X_flat_scaled = pre_pca_scaler.fit_transform(X_flat)

# Now apply PCA on properly scaled data
pca = PCA(n_components=0.95, random_state=42)
X_pca_flat = pca.fit_transform(X_flat_scaled)

n_components = X_pca_flat.shape[1]
print(f"Original features: {n_feat} -> PCA components: {n_components}")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.4f}")

X_pca = X_pca_flat.reshape(n_samples, n_steps, n_components)
print("New X shape after PCA:", X_pca.shape)

Original features: 25 -> PCA components: 14
Variance explained: 0.9521
New X shape after PCA: (49370, 72, 14)


In [9]:
# Cell 6: Per-ticker chronological split on PCA-reduced data
X_train_list, X_test_list, y_train_list, y_test_list = [], [], [], []

idx = 0
for t in tickers:
    n = ticker_counts[t]
    X_t = X_pca[idx:idx+n]
    y_t = y[idx:idx+n]
    split_idx = int(n * 0.8)

    X_train_list.append(X_t[:split_idx])
    X_test_list.append(X_t[split_idx:])
    y_train_list.append(y_t[:split_idx])
    y_test_list.append(y_t[split_idx:])
    idx += n

X_train = np.concatenate(X_train_list)
X_test = np.concatenate(X_test_list)
y_train = np.concatenate(y_train_list)
y_test = np.concatenate(y_test_list)

print("Train:", X_train.shape, "Test:", X_test.shape)

# Final scaling (on top of PCA components, standard practice)
n_features = X_train.shape[2]
scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, n_features))
X_train_scaled = scaler.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
X_test_scaled = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

print("Scaled Train:", X_train_scaled.shape, "Scaled Test:", X_test_scaled.shape)

Train: (39490, 72, 14) Test: (9880, 72, 14)
Scaled Train: (39490, 72, 14) Scaled Test: (9880, 72, 14)


In [11]:
# NEW CELL 7 — insert after Cell 6 (split/scale), before Cell 8 (training)
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

X_train_flat_t = torch.tensor(X_train_scaled.reshape(X_train_scaled.shape[0], -1), dtype=torch.float32)
X_test_flat_t = torch.tensor(X_test_scaled.reshape(X_test_scaled.shape[0], -1), dtype=torch.float32)
train_loader_flat = DataLoader(TensorDataset(X_train_flat_t, y_train_t), batch_size=128, shuffle=True)

class PriceFFN(nn.Module):
    def __init__(self, input_size, hidden_size=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(hidden_size, 32), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        x_flat = x.reshape(x.size(0), -1) if x.dim() > 2 else x
        return self.net(x_flat).squeeze(-1)

input_size = 72 * n_features  # 72 timesteps x number of PCA components
criterion = nn.BCEWithLogitsLoss()

print("Model class and tensors ready. input_size =", input_size)

Model class and tensors ready. input_size = 1008


In [12]:
# Cell 8 (fixed): Retrain with lower learning rate to avoid collapse
model_ffn = PriceFFN(input_size=input_size)
optimizer = optim.Adam(model_ffn.parameters(), lr=5e-4, weight_decay=1e-5)  # 10x lower than before
criterion = nn.BCEWithLogitsLoss()

best_acc, best_state = 0, None
for epoch in range(15):
    model_ffn.train()
    for X_batch, y_batch in train_loader_flat:
        optimizer.zero_grad()
        logits = model_ffn(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ffn.parameters(), max_norm=1.0)
        optimizer.step()

    model_ffn.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model_ffn(X_test_flat_t)) > 0.5).float()
        acc = (preds == y_test_t).float().mean().item()
        pred_up = (preds == 1).sum().item()
    if acc > best_acc:
        best_acc = acc
        best_state = copy.deepcopy(model_ffn.state_dict())
    print(f"Epoch {epoch+1}/15 | Test Acc: {acc:.4f} | Predicted 'up': {pred_up}/{len(preds)}")

model_ffn.load_state_dict(best_state)
print(f"\nBest test accuracy: {best_acc:.4f}")

Epoch 1/15 | Test Acc: 0.5041 | Predicted 'up': 7042/9880
Epoch 2/15 | Test Acc: 0.5138 | Predicted 'up': 5543/9880
Epoch 3/15 | Test Acc: 0.5064 | Predicted 'up': 6202/9880
Epoch 4/15 | Test Acc: 0.5023 | Predicted 'up': 7428/9880
Epoch 5/15 | Test Acc: 0.5039 | Predicted 'up': 6856/9880
Epoch 6/15 | Test Acc: 0.4995 | Predicted 'up': 5790/9880
Epoch 7/15 | Test Acc: 0.5050 | Predicted 'up': 6620/9880
Epoch 8/15 | Test Acc: 0.5041 | Predicted 'up': 6498/9880
Epoch 9/15 | Test Acc: 0.5029 | Predicted 'up': 5832/9880
Epoch 10/15 | Test Acc: 0.5068 | Predicted 'up': 5978/9880
Epoch 11/15 | Test Acc: 0.5033 | Predicted 'up': 5806/9880
Epoch 12/15 | Test Acc: 0.5047 | Predicted 'up': 6557/9880
Epoch 13/15 | Test Acc: 0.5030 | Predicted 'up': 6371/9880
Epoch 14/15 | Test Acc: 0.5075 | Predicted 'up': 5769/9880
Epoch 15/15 | Test Acc: 0.5035 | Predicted 'up': 6484/9880

Best test accuracy: 0.5138


In [13]:
# Cell 9: Check what the model is actually predicting
model_ffn.eval()
with torch.no_grad():
    test_preds = (torch.sigmoid(model_ffn(X_test_flat_t)) > 0.5).float()

print("Predicted class distribution:")
print("Predicted 1 (up):", (test_preds == 1).sum().item())
print("Predicted 0 (down):", (test_preds == 0).sum().item())
print("\nActual class distribution:")
print("Actual 1 (up):", (y_test_t == 1).sum().item())
print("Actual 0 (down):", (y_test_t == 0).sum().item())

Predicted class distribution:
Predicted 1 (up): 5543
Predicted 0 (down): 4337

Actual class distribution:
Actual 1 (up): 4937
Actual 0 (down): 4943


In [15]:
# Cell 10: Integrated Gradients via captum
from captum.attr import IntegratedGradients

model_ffn.eval()
ig = IntegratedGradients(model_ffn)

# Take same 20 test samples as before, for consistency
explain_sample_ig = X_test_flat_t[:20]
baseline_input = torch.zeros_like(explain_sample_ig)  # zero baseline (standard for IG)

attributions_baseline, delta = ig.attribute(
    explain_sample_ig, baseline_input, target=None, return_convergence_delta=True, n_steps=50
)

print("Baseline IG attributions shape:", attributions_baseline.shape)
print("Convergence delta (should be small):", delta.abs().mean().item())

Baseline IG attributions shape: torch.Size([20, 1008])
Convergence delta (should be small): 0.0006527584628202021


In [16]:
# Cell 11: Stability test with Integrated Gradients
np.random.seed(42)
noise = torch.normal(0, 0.001, explain_sample_ig.shape)
perturbed_input_ig = explain_sample_ig + noise

attributions_perturbed, _ = ig.attribute(
    perturbed_input_ig, baseline_input, target=None, return_convergence_delta=True, n_steps=50
)

diff_ig = (attributions_baseline - attributions_perturbed).abs().mean().item()
scale_ig = attributions_baseline.abs().mean().item()
instability_ig = diff_ig / (scale_ig + 1e-9)

print(f"Integrated Gradients — Mean absolute difference: {diff_ig:.6f}")
print(f"Integrated Gradients — Relative instability: {instability_ig*100:.2f}%")
print(f"\nComparison: SHAP instability was ~192-199% (from earlier experiments)")

Integrated Gradients — Mean absolute difference: 0.000007
Integrated Gradients — Relative instability: 0.60%

Comparison: SHAP instability was ~192-199% (from earlier experiments)


In [17]:
# Cell 12: Gradient-penalty regularized retrain on final dataset
def train_with_penalty(penalty_weight, epochs=15):
    model = PriceFFN(input_size=input_size)
    optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)  # keeping the fixed, lower LR
    best_acc, best_state = 0, None

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader_flat:
            X_batch.requires_grad_(True)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            if penalty_weight > 0:
                grads = torch.autograd.grad(logits.sum(), X_batch, create_graph=True)[0]
                grad_penalty = (grads.norm(dim=1) ** 2).mean()
                loss = loss + penalty_weight * grad_penalty

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            preds = (torch.sigmoid(model(X_test_flat_t)) > 0.5).float()
            acc = (preds == y_test_t).float().mean().item()
        if acc > best_acc:
            best_acc = acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, best_acc

model_penalty, acc_penalty = train_with_penalty(penalty_weight=0.05)
print(f"Gradient-penalty model — Test accuracy: {acc_penalty:.4f}")

Gradient-penalty model — Test accuracy: 0.5138


In [19]:
import shap

In [20]:
# Cell 13: SHAP stability check on the gradient-penalty model
model_penalty.eval()

def model_predict_penalty(x_flat):
    x_tensor = torch.tensor(x_flat, dtype=torch.float32)
    with torch.no_grad():
        return torch.sigmoid(model_penalty(x_tensor)).numpy()

background_flat_final = X_train_flat_t[:50].numpy()
explain_sample_final = X_test_flat_t[:20].numpy()

explainer_final = shap.KernelExplainer(model_predict_penalty, background_flat_final)
base_shap_final = np.array(explainer_final.shap_values(explain_sample_final[:5], nsamples=300))

perturbed_final = explain_sample_final[:5] + np.random.normal(0, 0.001, explain_sample_final[:5].shape)
pert_shap_final = np.array(explainer_final.shap_values(perturbed_final, nsamples=300))

diff_final = np.abs(base_shap_final - pert_shap_final).mean()
scale_final = np.abs(base_shap_final).mean()
instability_final = diff_final / (scale_final + 1e-9)

print(f"Gradient-penalty model — SHAP instability: {instability_final*100:.2f}%")
print(f"Compare to unregularized baseline: ~192-199%")

100%|██████████| 5/5 [00:01<00:00,  3.84it/s]

Gradient-penalty model — SHAP instability: 212.47%
Compare to unregularized baseline: ~192-199%


In [21]:
# Cell 14: Identify which original features dominate each PCA component
original_feature_names = [
    "open", "high", "low", "close",
    "sma_12", "sma_48", "rsi_14", "volatility_24", "momentum_6",
    "macd", "macd_signal", "macd_hist",
    "bb_width", "bb_position",
    "atr_14", "stoch_k", "stoch_d",
    "lag_return_3h", "lag_return_6h", "lag_return_12h", "lag_return_24h",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos"
]

component_labels = []
for comp_idx in range(pca.components_.shape[0]):
    loadings = pca.components_[comp_idx]
    top_2_idx = np.argsort(np.abs(loadings))[-2:][::-1]
    label = " + ".join([original_feature_names[i] for i in top_2_idx])
    component_labels.append(label)
    print(f"Component {comp_idx}: dominated by [{label}]")

Component 0: dominated by [bb_position + stoch_k]
Component 1: dominated by [close + high]
Component 2: dominated by [sma_12 + sma_48]
Component 3: dominated by [volatility_24 + bb_width]
Component 4: dominated by [macd_signal + macd]
Component 5: dominated by [dow_cos + dow_sin]
Component 6: dominated by [hour_cos + hour_sin]
Component 7: dominated by [hour_sin + hour_cos]
Component 8: dominated by [hour_sin + lag_return_3h]
Component 9: dominated by [dow_cos + dow_sin]
Component 10: dominated by [macd_hist + close]
Component 11: dominated by [open + macd_hist]
Component 12: dominated by [lag_return_24h + lag_return_12h]
Component 13: dominated by [high + low]


In [22]:
# Cell 15: Map technical feature names to plain-English descriptions
plain_language_map = {
    "open": "the starting price",
    "high": "the highest price reached",
    "low": "the lowest price reached",
    "close": "the closing price",
    "sma_12": "the short-term price trend (last 12 hours)",
    "sma_48": "the longer-term price trend (last 2 days)",
    "rsi_14": "how 'overbought' or 'oversold' the stock currently looks",
    "volatility_24": "how choppy or jumpy the price has been recently",
    "momentum_6": "how fast the price has been moving lately",
    "macd": "the overall trend strength",
    "macd_signal": "the trend strength (smoothed)",
    "macd_hist": "whether the trend is speeding up or slowing down",
    "bb_width": "how stretched the price is compared to its normal range",
    "bb_position": "whether the price is unusually high or low right now",
    "atr_14": "the typical size of recent price swings",
    "stoch_k": "how close the price is to its recent high or low point",
    "stoch_d": "how close the price is to its recent high or low point (smoothed)",
    "lag_return_3h": "how much the price moved in the last 3 hours",
    "lag_return_6h": "how much the price moved in the last 6 hours",
    "lag_return_12h": "how much the price moved in the last 12 hours",
    "lag_return_24h": "how much the price moved in the last day",
    "hour_sin": "what time of day it is",
    "hour_cos": "what time of day it is",
    "dow_sin": "what day of the week it is",
    "dow_cos": "what day of the week it is",
}

def plain_label(component_label):
    parts = component_label.split(" + ")
    plain_parts = [plain_language_map.get(p, p) for p in parts]
    # Remove duplicates (e.g. hour_sin + hour_cos both mean "time of day")
    unique_parts = list(dict.fromkeys(plain_parts))
    return " and ".join(unique_parts)

plain_component_labels = [plain_label(label) for label in component_labels]
for i, label in enumerate(plain_component_labels):
    print(f"Component {i}: {label}")

Component 0: whether the price is unusually high or low right now and how close the price is to its recent high or low point
Component 1: the closing price and the highest price reached
Component 2: the short-term price trend (last 12 hours) and the longer-term price trend (last 2 days)
Component 3: how choppy or jumpy the price has been recently and how stretched the price is compared to its normal range
Component 4: the trend strength (smoothed) and the overall trend strength
Component 5: what day of the week it is
Component 6: what time of day it is
Component 7: what time of day it is
Component 8: what time of day it is and how much the price moved in the last 3 hours
Component 9: what day of the week it is
Component 10: whether the trend is speeding up or slowing down and the closing price
Component 11: the starting price and whether the trend is speeding up or slowing down
Component 12: how much the price moved in the last day and how much the price moved in the last 12 hours
Comp

In [23]:
# Cell 16 (fixed): Correct order — scale per-timestep, then flatten
def predict_on_new_data(raw_ohlc_df):
    d = raw_ohlc_df.copy().reset_index(drop=True)
    d = add_indicators_v2(d)
    d = add_lag_and_time_features(d)

    price_cols = ["open", "high", "low", "close"]
    other_cols = [c for c in base_feature_cols if c not in price_cols]

    df_returns = d[price_cols].pct_change()
    df_other = d[other_cols]
    df_combined = pd.concat([df_returns, df_other], axis=1).dropna().reset_index(drop=True)

    if len(df_combined) < WINDOW:
        raise ValueError(f"Need at least {WINDOW} valid rows after feature calculation, got {len(df_combined)}")

    latest_window = df_combined.values[-WINDOW:]  # shape (72, 25)

    # Step 1: pre-PCA scaling (still shape 72, 25)
    latest_scaled_pre_pca = pre_pca_scaler.transform(latest_window)

    # Step 2: PCA (shape becomes 72, 14)
    latest_pca = pca.transform(latest_scaled_pre_pca)

    # Step 3: final scaling — apply DIRECTLY on (72, 14), no flattening yet
    latest_final_scaled = scaler.transform(latest_pca)  # shape stays (72, 14)

    # Step 4: NOW flatten for the model's flat input format
    x_flat = torch.tensor(latest_final_scaled.reshape(1, -1), dtype=torch.float32)  # shape (1, 1008)

    model_penalty.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model_penalty(x_flat)).item()
        prediction = "UP" if prob > 0.5 else "DOWN"

    x_flat_np = x_flat.numpy()
    shap_vals = explainer_final.shap_values(x_flat_np, nsamples=300)
    explanation = explain_prediction_simply(np.array(shap_vals), sample_idx=0)

    print(f"Prediction: {prediction} (confidence: {prob:.2%})")
    print(explanation)
    return prediction, prob, explanation

print("Prediction function fixed and ready")

Prediction function fixed and ready


In [24]:
# Cell 16: Editable synthetic OHLC data — adjust the 'close' values below to test different scenarios
import numpy as np
import pandas as pd

# EDIT THESE: 150 hourly closing prices — change any values to simulate different market behavior
# Example pattern below: a mostly upward trend with some noise. Replace with your own numbers if you like.
close_values = [
    150.00, 150.25, 150.10, 150.45, 150.60, 150.55, 150.80, 151.00, 150.90, 151.20,
    151.35, 151.10, 151.50, 151.70, 151.60, 151.90, 152.10, 151.95, 152.30, 152.50,
    152.40, 152.70, 152.90, 152.80, 153.10, 153.30, 153.20, 153.50, 153.70, 153.60,
    153.90, 154.10, 154.00, 154.30, 154.50, 154.40, 154.70, 154.90, 154.80, 155.10,
    155.30, 155.20, 155.50, 155.70, 155.60, 155.90, 156.10, 156.00, 156.30, 156.50,
] * 3  # repeats the pattern 3x to reach 150 hours — replace this with your own full list if you prefer

close_values = close_values[:150]  # trim to exactly 150 hours

close_prices = np.array(close_values)
open_prices = np.roll(close_prices, 1)
open_prices[0] = close_prices[0]

# EDIT THESE: how much "wiggle room" high/low have above/below open-close (currently small and realistic)
high_prices = np.maximum(open_prices, close_prices) + np.random.uniform(0.05, 0.3, len(close_prices))
low_prices = np.minimum(open_prices, close_prices) - np.random.uniform(0.05, 0.3, len(close_prices))

timestamps = pd.date_range(start="2025-01-01 09:00", periods=len(close_prices), freq="h")

new_stock_data = pd.DataFrame({
    "open_time": timestamps,
    "open": open_prices,
    "high": high_prices,
    "low": low_prices,
    "close": close_prices,
})

print(new_stock_data.shape)
new_stock_data.head(10)

(150, 5)


,open_time,open,high,low,close
0,2025-01-01 09:00:00,150.00,150.175416,149.701505,150.00
1,2025-01-01 10:00:00,150.00,150.453920,149.770665,150.25
2,2025-01-01 11:00:00,150.25,150.400626,150.024631,150.10
3,2025-01-01 12:00:00,150.10,150.524899,150.040249,150.45
4,2025-01-01 13:00:00,150.45,150.876404,150.359447,150.60
5,2025-01-01 14:00:00,150.60,150.753880,150.339305,150.55
6,2025-01-01 15:00:00,150.55,150.977168,150.449366,150.80
7,2025-01-01 16:00:00,150.80,151.110164,150.737833,151.00
8,2025-01-01 17:00:00,151.00,151.160979,150.807288,150.90
9,2025-01-01 18:00:00,150.90,151.291264,150.675855,151.20


In [27]:
# cell 17
def explain_prediction_simply(shap_vals_for_sample, sample_idx=0, top_n=4):
    vals = shap_vals_for_sample[sample_idx]
    n_comp = len(plain_component_labels)
    sorted_indices = np.argsort(np.abs(vals))[::-1]
    direction = "UP" if vals.sum() > 0 else "DOWN"

    reasons = []
    seen = set()
    for idx in sorted_indices:
        comp_idx = idx % n_comp
        label = plain_component_labels[comp_idx]
        if label not in seen:
            contribution = vals[idx]
            push_direction = "UP" if contribution > 0 else "DOWN"
            reasons.append({"label": label, "shap_value": contribution, "push": push_direction})
            seen.add(label)
        if len(reasons) == top_n:
            break
    return reasons

print("explain_prediction_simply ready")

explain_prediction_simply ready


In [28]:
# Cell 18: Get a prediction on your edited data
prediction, confidence, explanation = predict_on_new_data(new_stock_data)

100%|██████████| 1/1 [00:00<00:00,  3.14it/s]

Prediction: DOWN (confidence: 47.95%)
[{'label': 'the highest price reached and the lowest price reached', 'shap_value': np.float64(0.01900903085872899), 'push': 'UP'}, {'label': 'what day of the week it is', 'shap_value': np.float64(0.011431604895554285), 'push': 'UP'}, {'label': 'the short-term price trend (last 12 hours) and the longer-term price trend (last 2 days)', 'shap_value': np.float64(0.009260614535955393), 'push': 'UP'}, {'label': 'what time of day it is', 'shap_value': np.float64(-0.005850813003663153), 'push': 'DOWN'}]


In [29]:
# Cell 19: LLM-powered explanation generator using Groq (free tier)
!pip install groq --quiet

from groq import Groq

client = Groq(api_key="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")  # paste your free Groq key here

def generate_llm_reasoning(reasons, direction, confidence):
    reasons_text = "\n".join([
        f"- {r['label']} (SHAP contribution: {r['shap_value']:.6f}, pushed prediction toward {r['push']})"
        for r in reasons
    ])

    prompt = f"""You are explaining a machine learning model's stock price prediction to a reader with no technical background.

Model prediction: price will go {direction}
Model confidence: {confidence:.1%}

The model's top contributing factors, from a SHAP explainability analysis, are:
{reasons_text}

Write a short, honest explanation (3-4 sentences) that:
1. States the prediction and confidence level plainly.
2. Explains the top 2-3 factors in plain English, based ONLY on the data above -- do not invent or assume any factors not listed.
3. Notes clearly that {confidence:.1%} confidence is close to a coin flip, so this should not be treated as a strong signal.

Do not use technical jargon like "SHAP," "PCA," or "gradient." Do not exaggerate the model's reliability."""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",  # free, fast, high-quality Groq model
        messages=[{"role": "user", "content": prompt}],
        max_tokens=250,
    )
    return response.choices[0].message.content

print("Groq-based LLM reasoning function ready")

Groq-based LLM reasoning function ready


In [30]:
#  cell 20: Re-run this EXACT cell again now, to make sure the correct version is active
def explain_prediction_simply(shap_vals_for_sample, sample_idx=0, top_n=4):
    vals = shap_vals_for_sample[sample_idx]
    n_comp = len(plain_component_labels)

    sorted_indices = np.argsort(np.abs(vals))[::-1]

    direction = "UP" if vals.sum() > 0 else "DOWN"

    reasons = []
    seen = set()
    for idx in sorted_indices:
        comp_idx = idx % n_comp
        label = plain_component_labels[comp_idx]
        if label not in seen:
            contribution = vals[idx]
            push_direction = "UP" if contribution > 0 else "DOWN"
            reasons.append({
                "label": label,
                "shap_value": contribution,
                "push": push_direction
            })
            seen.add(label)
        if len(reasons) == top_n:
            break

    print(f"PREDICTION: {direction}")
    print(f"{'='*60}")
    print(f"Genuine contributing factors (ranked by influence strength):\n")
    for i, r in enumerate(reasons, 1):
        print(f"{i}. {r['label'].capitalize()}")
        print(f"   -> pushed prediction toward {r['push']}")
        print(f"   -> SHAP contribution strength: {r['shap_value']:.6f}\n")

    return reasons

In [31]:
# Cell 21: Full pipeline — predict, get structured reasons, then LLM-generated explanation
def predict_with_llm_reasoning(raw_ohlc_df):
    d = raw_ohlc_df.copy().reset_index(drop=True)
    d = add_indicators_v2(d)
    d = add_lag_and_time_features(d)

    price_cols = ["open", "high", "low", "close"]
    other_cols = [c for c in base_feature_cols if c not in price_cols]
    df_returns = d[price_cols].pct_change()
    df_other = d[other_cols]
    df_combined = pd.concat([df_returns, df_other], axis=1).dropna().reset_index(drop=True)

    if len(df_combined) < WINDOW:
        raise ValueError(f"Need at least {WINDOW} valid rows, got {len(df_combined)}")

    latest_window = df_combined.values[-WINDOW:]
    latest_scaled_pre_pca = pre_pca_scaler.transform(latest_window)
    latest_pca = pca.transform(latest_scaled_pre_pca)
    latest_final_scaled = scaler.transform(latest_pca)
    x_flat = torch.tensor(latest_final_scaled.reshape(1, -1), dtype=torch.float32)

    model_penalty.eval()
    with torch.no_grad():
        prob = torch.sigmoid(model_penalty(x_flat)).item()
        direction = "UP" if prob > 0.5 else "DOWN"

    shap_vals = explainer_final.shap_values(x_flat.numpy(), nsamples=300)
    reasons = explain_prediction_simply(np.array(shap_vals), sample_idx=0)

    llm_text = generate_llm_reasoning(reasons, direction, prob)

    print(f"PREDICTION: {direction} (confidence: {prob:.2%})\n")
    print("LLM-generated explanation:")
    print(llm_text)

    return direction, prob, llm_text

direction, confidence, llm_explanation = predict_with_llm_reasoning(new_stock_data)

100%|██████████| 1/1 [00:00<00:00,  2.51it/s]


PREDICTION: DOWN
Genuine contributing factors (ranked by influence strength):

1. What time of day it is and how much the price moved in the last 3 hours
   -> pushed prediction toward UP
   -> SHAP contribution strength: 0.014913

2. What time of day it is
   -> pushed prediction toward UP
   -> SHAP contribution strength: 0.010824

3. The short-term price trend (last 12 hours) and the longer-term price trend (last 2 days)
   -> pushed prediction toward DOWN
   -> SHAP contribution strength: -0.010437

4. The highest price reached and the lowest price reached
   -> pushed prediction toward UP
   -> SHAP contribution strength: 0.009764

PREDICTION: DOWN (confidence: 47.95%)

LLM-generated explanation:
The model is predicting that the stock price will go down, but it's only about 47.9% confident in this prediction. When making this prediction, the model considered factors like the time of day and recent price movements over the last few hours, which actually suggested the price might go